# Jaffle Shop ELT Pipeline

**Real-world example: dbt-style e-commerce data pipeline with column lineage**

This notebook demonstrates a realistic analytics engineering pipeline modeled after the
[dbt Jaffle Shop](https://github.com/dbt-labs/jaffle-shop) — the canonical example in the
analytics engineering community.

### Pipeline Architecture

```
Raw Sources          Staging              Intermediate           Marts
───────────          ───────              ────────────           ─────
raw.customers   →  stg_customers    ┐
                                     ├→  int_customer_orders  → customers (dim)
raw.orders      →  stg_orders       ┤
                                     ├→  int_order_payments   → orders (fact)
raw.payments    →  stg_payments     ┘
```

### What You'll Learn

1. Multi-stage ELT pipeline with 8 queries across 4 layers
2. Table-level dependency analysis and execution ordering
3. Column-level lineage tracing (backward and forward)
4. PII detection and metadata propagation
5. Impact analysis: "what breaks if this source column changes?"
6. Pipeline visualization at table and column levels

## 1. Define the Pipeline

We define 8 SQL queries organized into 4 layers, mimicking a real dbt project.
Each query is a `(query_id, sql)` tuple.

In [ ]:
from clgraph import Pipeline

queries = [
    # ── Layer 1: Staging (clean & rename raw sources) ───────────────────
    (
        "stg_customers",
        """
        CREATE TABLE staging.stg_customers AS
        SELECT
            id         AS customer_id,   -- Primary key [owner: identity-team]
            first_name,                  -- Customer first name [pii: true, owner: identity-team]
            last_name,                   -- Customer last name [pii: true, owner: identity-team]
            email                        -- Contact email [pii: true, owner: identity-team]
        FROM raw.customers
        """,
    ),
    (
        "stg_orders",
        """
        CREATE TABLE staging.stg_orders AS
        SELECT
            id         AS order_id,        -- Primary key [owner: commerce-team]
            user_id    AS customer_id,      -- FK to customers [owner: commerce-team]
            order_date,                     -- Date order was placed [owner: commerce-team]
            status                          -- Order status [owner: commerce-team, tags: enum]
        FROM raw.orders
        """,
    ),
    (
        "stg_payments",
        """
        CREATE TABLE staging.stg_payments AS
        SELECT
            id          AS payment_id,        -- Primary key [owner: finance-team]
            order_id,                          -- FK to orders [owner: finance-team]
            payment_method,                    -- e.g. credit_card, coupon [owner: finance-team, tags: enum]
            amount / 100.0 AS amount_dollars   -- Convert cents to dollars [owner: finance-team, tags: metric currency]
        FROM raw.payments
        """,
    ),
    # ── Layer 2: Intermediate (business logic) ─────────────────────────
    (
        "int_order_payments",
        """
        CREATE TABLE intermediate.int_order_payments AS
        SELECT
            o.order_id,
            o.customer_id,
            o.order_date,
            o.status            AS order_status,
            SUM(p.amount_dollars) AS order_total
        FROM staging.stg_orders o
        LEFT JOIN staging.stg_payments p ON o.order_id = p.order_id
        GROUP BY o.order_id, o.customer_id, o.order_date, o.status
        """,
    ),
    (
        "int_customer_orders",
        """
        CREATE TABLE intermediate.int_customer_orders AS
        SELECT
            customer_id,
            MIN(order_date)   AS first_order_date,
            MAX(order_date)   AS most_recent_order_date,
            COUNT(order_id)   AS number_of_orders
        FROM intermediate.int_order_payments
        GROUP BY customer_id
        """,
    ),
    # ── Layer 3: Marts (final business entities) ───────────────────────
    (
        "mart_orders",
        """
        CREATE TABLE marts.orders AS
        SELECT
            op.order_id,
            op.customer_id,
            op.order_date,
            op.order_status,
            op.order_total,
            CASE
                WHEN op.order_total >= 100 THEN 'large'
                WHEN op.order_total >= 25  THEN 'medium'
                ELSE 'small'
            END AS order_size
        FROM intermediate.int_order_payments op
        """,
    ),
    (
        "mart_customers",
        """
        CREATE TABLE marts.customers AS
        SELECT
            c.customer_id,
            c.first_name,
            c.last_name,
            c.email,
            co.first_order_date,
            co.most_recent_order_date,
            co.number_of_orders,
            COALESCE(ltv.customer_lifetime_value, 0) AS customer_lifetime_value
        FROM staging.stg_customers c
        LEFT JOIN intermediate.int_customer_orders co ON c.customer_id = co.customer_id
        LEFT JOIN (
            SELECT
                customer_id,
                SUM(order_total) AS customer_lifetime_value
            FROM intermediate.int_order_payments
            GROUP BY customer_id
        ) ltv ON c.customer_id = ltv.customer_id
        """,
    ),
    # ── Layer 4: Aggregated reporting ──────────────────────────────────
    (
        "rpt_customer_segments",
        """
        CREATE TABLE reporting.customer_segments AS
        SELECT
            CASE
                WHEN customer_lifetime_value >= 200 THEN 'champion'
                WHEN customer_lifetime_value >= 100 THEN 'loyal'
                WHEN number_of_orders >= 3           THEN 'repeat'
                WHEN number_of_orders = 1            THEN 'new'
                ELSE 'inactive'
            END AS segment,
            COUNT(*)                         AS customer_count,
            AVG(customer_lifetime_value)      AS avg_ltv,
            AVG(number_of_orders)             AS avg_orders
        FROM marts.customers
        GROUP BY 1
        """,
    ),
]

pipeline = Pipeline(queries, dialect="bigquery")

print(f"Pipeline built: {len(pipeline.table_graph.queries)} queries, "
      f"{len(pipeline.table_graph.tables)} tables, "
      f"{len(pipeline.columns)} columns")

## 2. Table-Level Dependencies

View the execution order — which tables must be built before others.
Source tables (no upstream query) are listed first.

In [ ]:
print("TABLE EXECUTION ORDER")
print("=" * 50)
for i, table in enumerate(pipeline.table_graph.get_execution_order(), 1):
    node = pipeline.table_graph.tables[str(table)]
    if node.is_source:
        print(f"  {i}. {table}  (source)")
    else:
        query = pipeline.table_graph.queries.get(node.created_by)
        deps = ', '.join(sorted(query.source_tables)) if query and query.source_tables else ''
        print(f"  {i}. {table}")
        if deps:
            print(f"     <- {deps}")

print()
print(f"Source tables: {[str(t) for t in pipeline.table_graph.get_source_tables()]}")
print(f"Final tables:  {[str(t) for t in pipeline.table_graph.get_final_tables()]}")

### Visualize Table Dependencies

In [ ]:
import shutil

from clgraph import visualize_table_dependencies

if shutil.which("dot") is None:
    print("Graphviz not installed. Install with: brew install graphviz")
else:
    display(visualize_table_dependencies(pipeline.table_graph))

## 3. Backward Lineage — Where Does a Column Come From?

Trace `marts.customers.customer_lifetime_value` back to its raw sources.
This is the most common lineage question: "what raw data feeds this metric?"

In [ ]:
print("BACKWARD LINEAGE: marts.customers.customer_lifetime_value")
print("=" * 60)

sources = pipeline.trace_column_backward("marts.customers", "customer_lifetime_value")
print(f"\nTraces back to {len(sources)} source column(s):")
for s in sources:
    print(f"  -> {s.table_name}.{s.column_name}  ({s.node_type})")

print("\n--- Full path (with intermediates) ---")
nodes, edges = pipeline.trace_column_backward_full("marts.customers", "customer_lifetime_value")
for node in nodes:
    print(f"  {node.table_name}.{node.column_name}  [{node.layer or 'n/a'}]")

### Visualize the Lineage Path

In [ ]:
from clgraph import visualize_lineage_path

if shutil.which("dot") is not None:
    nodes, edges = pipeline.trace_column_backward_full("marts.customers", "customer_lifetime_value")
    display(visualize_lineage_path(nodes, edges, is_backward=True))

## 4. Forward Lineage — Impact Analysis

Answer: "If `raw.payments.amount` changes, which downstream tables and columns are affected?"

This is critical for schema migration planning and data governance.

In [ ]:
print("FORWARD LINEAGE (Impact Analysis): raw.payments.amount")
print("=" * 60)

impacts = pipeline.trace_column_forward("raw.payments", "amount")
print(f"\nImpacts {len(impacts)} downstream column(s):")

# Group by table for readability
by_table = {}
for col in impacts:
    by_table.setdefault(col.table_name, []).append(col.column_name)

for table, cols in sorted(by_table.items()):
    print(f"  {table}:")
    for col in sorted(cols):
        print(f"    -> {col}")

In [ ]:
# Also trace impact of a source column rename
print("IMPACT ANALYSIS: raw.orders.user_id")
print("=" * 60)
print("(renamed to customer_id in staging)\n")

impacts = pipeline.trace_column_forward("raw.orders", "user_id")
by_table = {}
for col in impacts:
    by_table.setdefault(col.table_name, []).append(col.column_name)

for table, cols in sorted(by_table.items()):
    print(f"  {table}: {', '.join(sorted(cols))}")

## 5. PII Detection & Metadata Propagation

We annotated PII columns in the staging layer via SQL comments.
Now propagate that metadata through the entire pipeline to find
every downstream table that touches personal data.

In [ ]:
pipeline.propagate_all_metadata()

print("\nPII COLUMNS ACROSS THE PIPELINE")
print("=" * 50)
pii_cols = pipeline.get_pii_columns()
by_table = {}
for col in pii_cols:
    by_table.setdefault(col.table_name, []).append(col.column_name)

for table, cols in sorted(by_table.items()):
    print(f"  {table}:")
    for col in sorted(cols):
        print(f"    ! {col}")

print(f"\nTotal PII columns: {len(pii_cols)}")
print("\nTables containing PII:")
for table in sorted(by_table.keys()):
    print(f"  - {table}")

## 6. Ownership Analysis

Who owns what? Useful for routing data quality alerts.

In [ ]:
print("COLUMN OWNERSHIP")
print("=" * 50)

for team in ["identity-team", "commerce-team", "finance-team"]:
    owned = pipeline.get_columns_by_owner(team)
    tables = sorted({c.table_name for c in owned})
    print(f"\n  {team} ({len(owned)} columns):")
    for t in tables:
        cols = [c.column_name for c in owned if c.table_name == t]
        print(f"    {t}: {', '.join(sorted(cols))}")

## 7. Column Detail — Final Mart Tables

Inspect what's in each mart table: expressions, metadata, and lineage depth.

In [ ]:
print("MART TABLE: marts.customers")
print("=" * 50)
for col in sorted(pipeline.columns.values(), key=lambda c: c.full_name):
    if col.table_name == "marts.customers":
        pii_flag = " [PII]" if col.pii else ""
        owner = f" ({col.owner})" if col.owner else ""
        print(f"  {col.column_name}: {col.expression}{pii_flag}{owner}")

print()
print("MART TABLE: marts.orders")
print("=" * 50)
for col in sorted(pipeline.columns.values(), key=lambda c: c.full_name):
    if col.table_name == "marts.orders":
        owner = f" ({col.owner})" if col.owner else ""
        print(f"  {col.column_name}: {col.expression}{owner}")

print()
print("REPORTING TABLE: reporting.customer_segments")
print("=" * 50)
for col in sorted(pipeline.columns.values(), key=lambda c: c.full_name):
    if col.table_name == "reporting.customer_segments":
        print(f"  {col.column_name}: {col.expression}")

## 8. Full Pipeline Visualization

Column-level lineage across all 8 queries — showing exactly how data flows
from raw sources through staging, intermediate, and mart layers.

In [ ]:
from clgraph import visualize_pipeline_lineage

if shutil.which("dot") is None:
    print("Graphviz not installed. Install with: brew install graphviz")
else:
    print("Full Pipeline Column Lineage:")
    display(visualize_pipeline_lineage(pipeline.column_graph))

In [ ]:
# Simplified view: collapse CTEs/subqueries, show only table-to-table edges
if shutil.which("dot") is not None:
    print("Simplified Lineage (Source -> Final):")
    display(visualize_pipeline_lineage(pipeline.column_graph.to_simplified()))

## 9. Export Pipeline Lineage

Export to JSON for integration with data catalogs or governance tools.

In [ ]:
import json

from clgraph import JSONExporter

export_data = JSONExporter.export(pipeline)

print(f"Exported: {len(export_data.get('columns', []))} columns, "
      f"{len(export_data.get('edges', []))} edges")

# Show a sample column entry
print("\nSample column entry:")
sample = next(
    (c for c in export_data.get("columns", [])
     if c.get("full_name", "").endswith("customer_lifetime_value")),
    None,
)
if sample:
    print(json.dumps(sample, indent=2))

# Show a sample edge
print("\nSample edge entry:")
sample_edge = next(
    (e for e in export_data.get("edges", [])
     if "order_total" in e.get("to_column", "")),
    None,
)
if sample_edge:
    print(json.dumps(sample_edge, indent=2))

## 10. Graph Object Summary

Quick overview of the two graph structures available for programmatic access.

In [ ]:
print("PIPELINE GRAPH SUMMARY")
print("=" * 50)

tg = pipeline.table_graph
cg = pipeline.column_graph

print(f"\n  Table Graph (TableDependencyGraph):")
print(f"    Tables:  {len(tg.tables)}")
print(f"    Queries: {len(tg.queries)}")
print(f"    Sources: {[str(t) for t in tg.get_source_tables()]}")
print(f"    Finals:  {[str(t) for t in tg.get_final_tables()]}")

print(f"\n  Column Graph (PipelineLineageGraph):")
print(f"    Columns: {len(cg.columns)}")
print(f"    Edges:   {len(cg.edges)}")
print(f"    Source columns:  {len(cg.get_source_columns())}")
print(f"    Final columns:   {len(cg.get_final_columns())}")

print("\nPipeline analysis complete!")